In [31]:
import json
import os
import random
import shutil

import pandas as pd

In [2]:
img_dir = "E:/Research/simplify-me/simplify_me_dataset/images"
meta_path = "E:/Research/simplify-me/simplify_me_dataset/meta.json"

save_dir = "E:/Research/simplify-me-dataset-eval-data/"

os.makedirs(save_dir, exist_ok=True)

In [3]:
def load_data():
    with open(meta_path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [4]:
def generate_unique_id(benchmark, img_id):
    return f'{benchmark}-{img_id}'

In [6]:
def get_and_preprocess_data(data, split, ds_size):
    processed_data = []

    data = data[split]

    data = sorted(
        data,
        key=lambda x: float(x['dpo']['sle_delta']),
        reverse=True
    )

    benchmark_count = {}

    benchmark_process = {}
    new_data = []
    if ds_size != -1:
        if split == 'train':
            data = data[:ds_size]
        else:
            for item in data:
                if item['benchmark'] not in benchmark_process:
                    benchmark_process[item['benchmark']] = 0

                if benchmark_process[item['benchmark']] >= 10:
                    continue

                new_data.append(item)
                benchmark_process[item['benchmark']] += 1

            data = new_data

    print(f"split: {split}, data length: {len(data)}, min sle delta: {data[-1]['dpo']['sle_delta']}")

    for item in data:

        if item['benchmark'] not in benchmark_count:
            benchmark_count[item['benchmark']] = 0

            os.makedirs(os.path.join(save_dir, split, 'images', item['benchmark']), exist_ok=True)

        benchmark_count[item['benchmark']] += 1

        item['unique_id'] = generate_unique_id(item['benchmark'], item['id'])

        processed_data.append(item)

    print(benchmark_count)
    filename = f'annotations-{split}{'' if split != 'train' else '-full' if ds_size == -1 else '-' + str(ds_size)}.json'

    with open(os.path.join(save_dir, split, filename), 'w', encoding='utf-8') as fp:
        json.dump(processed_data, fp, indent=4, ensure_ascii=False)

    return filename

In [8]:
raw_data = load_data()
max_size = [-1, 10000, 5000, 1000]

for sz in max_size:
    # fn = get_and_preprocess_data(raw_data, 'train', sz)
    fn = get_and_preprocess_data(raw_data, 'test', 80)

split: test, data length: 80, min sle delta: 3.7560667991638184
{'viswiz_train': 10, 'flickr30k_val': 10, 'coco_2014_train': 10, 'coco_2014_val': 10, 'viswiz_val': 10, 'nocaps_test': 10, 'textcaps_train': 10, 'textcaps_val': 10}
split: test, data length: 80, min sle delta: 3.7560667991638184
{'viswiz_train': 10, 'flickr30k_val': 10, 'coco_2014_train': 10, 'coco_2014_val': 10, 'viswiz_val': 10, 'nocaps_test': 10, 'textcaps_train': 10, 'textcaps_val': 10}
split: test, data length: 80, min sle delta: 3.7560667991638184
{'viswiz_train': 10, 'flickr30k_val': 10, 'coco_2014_train': 10, 'coco_2014_val': 10, 'viswiz_val': 10, 'nocaps_test': 10, 'textcaps_train': 10, 'textcaps_val': 10}
split: test, data length: 80, min sle delta: 3.7560667991638184
{'viswiz_train': 10, 'flickr30k_val': 10, 'coco_2014_train': 10, 'coco_2014_val': 10, 'viswiz_val': 10, 'nocaps_test': 10, 'textcaps_train': 10, 'textcaps_val': 10}


## Test split

### Image copying

In [9]:
with open(os.path.join(save_dir, 'test', 'annotations-test.json'), 'r', encoding='utf-8') as test_fp:
    test_data = json.load(test_fp)

In [10]:
for data in test_data:
    raw_img_path = os.path.join(img_dir, data['benchmark'], data['image']['file_name'])
    target_img_path = os.path.join(save_dir, 'test', 'images', data['benchmark'], data['image']['file_name'])

    shutil.copy(raw_img_path, target_img_path)

### Generated captions

In [11]:
test_data_ids = {}

for data in test_data:
    test_data_ids[data['unique_id']] = data

In [12]:
generated_caption_folder = "E:/Research/gen-captions/gen-captions/scored"

In [13]:
experiments = {
    'gemma3-3-epoch-1000-trained.json': 'gemma3-trained-1000.json',
    'gemma3-3-epoch-10000-trained.json': 'gemma3-trained-10000.json',
    'gemma3-3-epoch-full-trained.json': 'gemma3-trained-full.json',
    'gemma3-base.json': 'gemma3-zeroshot.json',
    'gemma3-sft.json': 'gemma3-sft.json',
    'Llama-3.2-11B-Vision-base.json': 'llama32-zeroshot.json',
    'Llama-32-11B-Vision-1000-trained.json': 'llama32-trained-1000.json',
    'Llama-32-11B-Vision-10000-trained.json': 'llama32-trained-10000.json',
    'Llama-32-11B-Vision-full-trained.json': 'llama32-trained-full.json',
    'Llama-3.2-sft.json': 'llama32-sft.json',
    'Qwen2-VL-2B-Instruct-1000-trained.json': 'qwen2-2b-trained-1000.json',
    'Qwen2-VL-2B-Instruct-10000-trained.json': 'qwen2-2b-trained-10000.json',
    'Qwen2-VL-2B-Instruct-base.json': 'qwen2-2b-zeroshot.json',
    'Qwen2-VL-2B-Instruct-full-trained.json': 'qwen2-2b-trained-full.json',
    'Qwen2-VL-2B-sft.json': 'qwen2-2b-sft.json',
    'Qwen2-VL-7B-Instruct-1000-trained.json': 'qwen2-7b-trained-1000.json',
    'Qwen2-VL-7B-Instruct-10000-trained.json': 'qwen2-7b-trained-10000.json',
    'Qwen2-VL-7B-Instruct-base.json': 'qwen2-7b-zeroshot.json',
    'Qwen2-VL-7B-Instruct-full-trained.json': 'qwen2-7b-trained-full.json',
    'Qwen2-VL-7B-sft.json': 'qwen2-7b-sft.json',
    'Qwen25-VL-7B-Instruct-1000-trained.json': 'qwen25-7b-trained-1000.json',
    'Qwen25-VL-7B-Instruct-10000-trained.json': 'qwen25-7b-trained-10000.json',
    'Qwen25-VL-7B-Instruct-full-trained.json': 'qwen25-7b-trained-full.json',
    'Qwen25-VL-7B-Instruct-base.json': 'qwen25-7b-zeroshot.json',
    'Qwen25-VL-7B-sft.json': 'qwen25-7b-sft.json'
}

In [16]:
generated_caption_files = os.listdir(generated_caption_folder)

for generated_caption_file in generated_caption_files:
    print(generated_caption_file)
    with open(os.path.join(generated_caption_folder, generated_caption_file), 'r', encoding='utf-8') as fp:
        generated_caption = json.load(fp)

    selected_generated_captions = []
    all_generated_captions = []

    for item in generated_caption:
        compiled_item = {
            'unique_id': generate_unique_id(item['benchmark'], item['id']),
            'id': item['id'],
            'benchmark': item['benchmark'],
            'file_name': item['images'][0].split('/')[-1],
            'instruction': item['instruction'],
            'generated_caption': item['model_output'],
            'sle_score': item['sle_score']['sle'][0],
        }

        all_generated_captions.append(compiled_item)

        if generate_unique_id(item['benchmark'], item['id']) in test_data_ids:
            selected_generated_captions.append(compiled_item)

    with open(os.path.join(save_dir, 'test', 'all_generated_captions', experiments[generated_caption_file]), 'w',
              encoding='utf-8') as fp:
        json.dump(all_generated_captions, fp, indent=5)

    if '10000' in generated_caption_file or 'base' in generated_caption_file or 'sft' in generated_caption_file:
        with open(os.path.join(save_dir, 'test', 'selected_generated_captions', experiments[generated_caption_file]),
                  'w', encoding='utf-8') as fp:
            json.dump(selected_generated_captions, fp, indent=5)

gemma3-3-epoch-1000-trained.json
gemma3-3-epoch-10000-trained.json
gemma3-3-epoch-full-trained.json
gemma3-base.json
gemma3-sft.json
Llama-3.2-11B-Vision-base.json
Llama-3.2-sft.json
Llama-32-11B-Vision-1000-trained.json
Llama-32-11B-Vision-10000-trained.json
Llama-32-11B-Vision-full-trained.json
Qwen2-VL-2B-Instruct-1000-trained.json
Qwen2-VL-2B-Instruct-10000-trained.json
Qwen2-VL-2B-Instruct-base.json
Qwen2-VL-2B-Instruct-full-trained.json
Qwen2-VL-2B-sft.json
Qwen2-VL-7B-Instruct-1000-trained.json
Qwen2-VL-7B-Instruct-10000-trained.json
Qwen2-VL-7B-Instruct-base.json
Qwen2-VL-7B-Instruct-full-trained.json
Qwen2-VL-7B-sft.json
Qwen25-VL-7B-Instruct-1000-trained.json
Qwen25-VL-7B-Instruct-10000-trained.json
Qwen25-VL-7B-Instruct-base.json
Qwen25-VL-7B-Instruct-full-trained.json
Qwen25-VL-7B-sft.json


In [17]:
trained_map = [
    {
        'experiment': 'gemma3',
        'zero_shot': 'gemma3-zeroshot.json',
        'sft': 'gemma3-sft.json',
        'trained': 'gemma3-trained-10000.json'
    },
    {
        'experiment': 'llama3.2',
        'zero_shot': 'llama32-zeroshot.json',
        'sft': 'llama32-sft.json',
        'trained': 'llama32-trained-10000.json'
    },
    {
        'experiment': 'qwen2-2b',
        'zero_shot': 'qwen2-2b-zeroshot.json',
        'sft': 'qwen2-2b-sft.json',
        'trained': 'qwen2-2b-trained-10000.json'
    },
    {
        'experiment': 'qwen2-7b',
        'zero_shot': 'qwen2-7b-zeroshot.json',
        'sft': 'qwen2-7b-sft.json',
        'trained': 'qwen2-7b-trained-10000.json'
    },
    {
        'experiment': 'qwen2.5-7b',
        'zero_shot': 'qwen25-7b-zeroshot.json',
        'sft': 'qwen25-7b-sft.json',
        'trained': 'qwen25-7b-trained-10000.json'
    },
]

In [33]:
for map in trained_map:
    df_zero_shot = pd.read_json(os.path.join(save_dir, "test", "selected_generated_captions", map['zero_shot']))
    df_sft = pd.read_json(os.path.join(save_dir, "test", "selected_generated_captions", map['sft']))
    df_trained = pd.read_json(os.path.join(save_dir, "test", "selected_generated_captions", map['trained']))

    zero_shot_vs_dpo = []
    sft_vs_dpo = []

    for i in range(len(df_zero_shot)):
        if df_zero_shot['unique_id'][i] != df_sft['unique_id'][i] and df_zero_shot['unique_id'][i] != df_trained['unique_id'][i]:
            print("Doesnt match")

        random_number = random.random()

        if random_number <= 0.5:
            caption_key = 'base'
        else:
            caption_key = 'trained'

        zero_shot_vs_dpo.append({
            'unique_id': df_zero_shot['unique_id'][i],
            'benchmark': df_zero_shot['benchmark'][i],
            'file_path': f"{df_zero_shot['benchmark'][i]}/{df_zero_shot['file_name'][i]}",
            'caption_key': caption_key,
            'caption_1': df_zero_shot['generated_caption'][i] if caption_key == 'base' else df_trained['generated_caption'][i],
            'caption_2': df_trained['generated_caption'][i] if caption_key == 'base' else df_zero_shot['generated_caption'][i],
            'preferred_caption': "",
            'comment': ""
        })

        sft_vs_dpo.append({
            'unique_id': df_zero_shot['unique_id'][i],
            'benchmark': df_zero_shot['benchmark'][i],
            'caption_key': caption_key,
            'file_path': f"{df_zero_shot['benchmark'][i]}/{df_zero_shot['file_name'][i]}",
            'caption_1': df_sft['generated_caption'][i] if caption_key == 'base' else df_trained['generated_caption'][i],
            'caption_2': df_trained['generated_caption'][i] if caption_key == 'base' else df_sft['generated_caption'][i],
        })

    pd.DataFrame(zero_shot_vs_dpo).to_csv(os.path.join(save_dir, "test", "qualitative", f"{map['experiment']}_zero_shot_vs_dpo.csv"), index=False)
    pd.DataFrame(sft_vs_dpo).to_csv(os.path.join(save_dir, "test", "qualitative", f"{map['experiment']}_sft_vs_dpo.csv"), index=False)

In [14]:
import numpy as np
from nltk import word_tokenize

# nltk.download('punkt_tab')

for expt in zeroshot_trained_map:
    zeroshot_file = os.path.join(save_dir, 'test', 'selected_generated_captions', expt['zero_shot'])
    sft_file = os.path.join(save_dir, 'test', 'selected_generated_captions', expt['sft'])
    trained_file = os.path.join(save_dir, 'test', 'selected_generated_captions', expt['trained'])

    zeroshot_lens = []
    trained_lens = []
    sft_lens = []

    with open(zeroshot_file, 'r', encoding='utf-8') as fp:
        zeroshot_data = json.load(fp)

    with open(sft_file, 'r', encoding='utf-8') as fp:
        sft_data = json.load(fp)

    with open(trained_file, 'r', encoding='utf-8') as fp:
        trained_data = json.load(fp)

    for item in zeroshot_data:
        zeroshot_lens.append(len(word_tokenize(item['generated_caption'])))

    for item in sft_data:
        if item['generated_caption'] != '':
            # print(len(item['generated_caption']))
            sft_lens.append(len(word_tokenize(item['generated_caption'])))

    for item in trained_data:
        if item['generated_caption'] != '':
            trained_lens.append(len(word_tokenize(item['generated_caption'])))

    print(
        f"{expt['experiment']}\tZeroshot: {np.mean(zeroshot_lens):.2f}\tSFT: {np.mean(sft_lens):.2f}\tTrained: {np.mean(trained_lens):.2f}. Ratio: {np.mean(trained_lens) / np.mean(zeroshot_lens):.2f}")

gemma3	Zeroshot: 94.07	SFT: 186.90	Trained: 68.92. Ratio: 0.73
llama3.2	Zeroshot: 181.39	SFT: 13.32	Trained: 39.59. Ratio: 0.22
qwen2-2b	Zeroshot: 45.08	SFT: 9.38	Trained: 12.72. Ratio: 0.28
qwen2-7b	Zeroshot: 47.58	SFT: 13.38	Trained: 8.80. Ratio: 0.18
qwen2.5-7b	Zeroshot: 75.33	SFT: 10.52	Trained: 52.98. Ratio: 0.70


In [15]:
import numpy as np
from nltk import word_tokenize

# nltk.download('punkt_tab')

for expt in zeroshot_trained_map:
    zeroshot_file = os.path.join(save_dir, 'test', 'all_generated_captions', expt['zero_shot'])
    trained_file = os.path.join(save_dir, 'test', 'all_generated_captions', expt['trained'])

    zeroshot_lens = []
    trained_lens = []

    with open(zeroshot_file, 'r', encoding='utf-8') as fp:
        zeroshot_data = json.load(fp)

    with open(trained_file, 'r', encoding='utf-8') as fp:
        trained_data = json.load(fp)

    for item in zeroshot_data:
        zeroshot_lens.append(len(word_tokenize(item['generated_caption'])))

    for item in trained_data:
        trained_lens.append(len(word_tokenize(item['generated_caption'])))

    print(
        f"{expt['experiment']}\tZeroshot: {np.mean(zeroshot_lens):.2f}\tTrained: {np.mean(trained_lens):.2f}. Ratio: {np.mean(trained_lens) / np.mean(zeroshot_lens):.2f}")

gemma3	Zeroshot: 89.66	Trained: 66.36. Ratio: 0.74
llama3.2	Zeroshot: 176.60	Trained: 42.15. Ratio: 0.24
qwen2-2b	Zeroshot: 36.67	Trained: 12.16. Ratio: 0.33
qwen2-7b	Zeroshot: 40.91	Trained: 8.34. Ratio: 0.20
qwen2.5-7b	Zeroshot: 70.58	Trained: 51.20. Ratio: 0.73
